# 6. HTEM conductivity: algorithm selection and experiment

**Workshop focus:** after data carpentry has been handled, use an experimental materials dataset to explore algorithm choice. The target is `log10_conductivity`; the input is a fixed-grid optical-transmittance spectrum.

This does not claim a process emulator. It is a controlled comparison of candidate models on one held-out experimental test set. The question for an LLM is not merely “write a neural network,” but “which methods should we try and how will we evaluate them honestly?”


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_FILE = Path('data/Filtered_Libraries_Data_wTauc_interpolated_trans.parquet')
raw = pd.read_parquet(DATA_FILE)
print(f'Prepared HTEM rows: {len(raw):,}')
print('Available spectra:', raw['interpolated_transmittances'].notna().sum())
display(raw[['id', 'fpm_conductivity', 'log10_conductivity', 'opt_direct_bandgap']].head())


In [ ]:
# The prepared table already contains spectra interpolated onto a common 500-point wavelength grid.
def usable_spectrum(values):
    if values is None:
        return False
    vector = np.asarray(values, dtype=float)
    return vector.ndim == 1 and len(vector) == 500 and np.isfinite(vector).all()

data = raw.dropna(subset=['id', 'log10_conductivity', 'interpolated_transmittances']).copy()
data = data[data['interpolated_transmittances'].map(usable_spectrum)].copy()
data = data.drop_duplicates(subset='id').reset_index(drop=True)
X = np.vstack(data['interpolated_transmittances'].map(lambda values: np.asarray(values, dtype=float)))
y = data['log10_conductivity'].to_numpy(dtype=float)

print(f'Rows after audit: {len(data):,}')
print(f'Feature matrix: {X.shape}; target range: {y.min():.2f} to {y.max():.2f} log10(S/cm)')
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.hist(y, bins=40, edgecolor='white')
ax.set(xlabel='log10 conductivity (S/cm)', ylabel='Samples', title='HTEM conductivity target distribution')
plt.show()


## Prompt: plan a model experiment

    > I have 5,000+ experimental samples, each with a 500-point optical-transmittance spectrum, and I want to predict log10 electrical conductivity. Propose three complementary regression algorithms appropriate for this dataset. Give a leakage-free train/validation/test experiment plan, including scaling, dimensionality reduction if useful, hyperparameter selection, MAE/RMSE/R² metrics, and a parity plot. Explain why a model that wins on the validation set must still be evaluated only once on the test set.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.40, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.50, random_state=SEED)

def spectral_pipeline(regressor):
    return make_pipeline(StandardScaler(), PCA(n_components=30, random_state=SEED), regressor)

candidates = {
    'Ridge regression': spectral_pipeline(Ridge(alpha=10.0)),
    'Random forest': spectral_pipeline(RandomForestRegressor(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=SEED)),
    'Gradient boosting': spectral_pipeline(HistGradientBoostingRegressor(max_iter=200, learning_rate=0.06, l2_regularization=1.0, random_state=SEED)),
}

validation_rows = []
for name, model in candidates.items():
    model.fit(X_train, y_train)
    predicted = model.predict(X_val)
    validation_rows.append({
        'model': name,
        'MAE': mean_absolute_error(y_val, predicted),
        'RMSE': mean_squared_error(y_val, predicted) ** 0.5,
        'R²': r2_score(y_val, predicted),
    })

validation = pd.DataFrame(validation_rows).sort_values('MAE').reset_index(drop=True)
display(validation.round(3))


In [ ]:
# Pick using validation only, then refit once using train + validation and report the untouched test score.
winner_name = validation.iloc[0]['model']
winner = candidates[winner_name]
winner.fit(np.vstack([X_train, X_val]), np.concatenate([y_train, y_val]))
test_prediction = winner.predict(X_test)

test_metrics = pd.DataFrame([{
    'selected model': winner_name,
    'test MAE': mean_absolute_error(y_test, test_prediction),
    'test RMSE': mean_squared_error(y_test, test_prediction) ** 0.5,
    'test R²': r2_score(y_test, test_prediction),
}])
display(test_metrics.round(3))

limits = [min(y_test.min(), test_prediction.min()), max(y_test.max(), test_prediction.max())]
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_test, test_prediction, s=14, alpha=0.55)
ax.plot(limits, limits, 'k--')
ax.set(xlabel='Measured log10 conductivity (S/cm)', ylabel='Predicted log10 conductivity (S/cm)',
       title=f'Untouched test set: {winner_name}')
ax.grid(alpha=0.2)
plt.show()


## Takeaway

The comparison is deliberately modest: a linear baseline and two nonlinear tree ensembles. A next iteration might add composition, XRD, deposition metadata, or a neural architecture—but only after deciding how to validate the additional representations and whether they reduce the number of usable samples.
